In [ ]:
! pip install llama-index-llms-google-genai llama-index

In [ ]:
! pip install llama-index-embeddings-google-genai

In [ ]:
!pip install patool

In [ ]:
import patoolib
patoolib.extract_archive("/content/data.zip")

# Chat Models (čata modeļi)

In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata
import asyncio

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

# Run async completion properly in Colab
resp = await llm.acomplete("Write a poem about a magic backpack")
print(resp)


In [5]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.llms import ChatMessage
from google.colab import userdata
import asyncio

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

messages = [
    ChatMessage(role="user", content="Hello friend!"),
    ChatMessage(role="assistant", content="Yarr what is shakin' matey?"),
    ChatMessage(role="user", content="Help me decide what to have for dinner."),
]

resp = await llm.achat(messages)  # <-- use `.achat()` instead of `.chat()`
print(resp)


assistant: Alright, let's figure out dinner! To give you the best recommendation, I need a little more information. Tell me:

1.  **What kind of food are you in the mood for?** (e.g., Italian, Mexican, Asian, American comfort food, something healthy, etc.)
2.  **How much time do you have to cook?** (e.g., Quick and easy - 30 minutes or less? Got some time to spare - an hour or more?)
3.  **What ingredients do you already have on hand?** (Mention any proteins, veggies, or pantry staples you want to use up.)
4.  **Are there any dietary restrictions or preferences?** (e.g., vegetarian, vegan, gluten-free, low-carb, allergies, etc.)
5.  **How adventurous are you feeling?** (e.g., Stick to the classics? Willing to try something new?)

Once I have this info, I can give you some personalized suggestions!



In [6]:
from llama_index.core.llms import ChatMessage

from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

import asyncio

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = GoogleGenAI(
    model="models/gemini-2.5-flash",
    api_key=GOOGLE_API_KEY
)

messages = []  # Initialize an empty list to store messages

while True:
    text_input = input("User: ")
    if text_input == "exit":
        break

    # Create a ChatMessage object for the user's input
    user_message = ChatMessage(role="user", content=text_input)
    messages.append(user_message)  # Add user message to history

    response = await llm.achat(messages)  # Pass the message history to llm.chat
    print(f"Agent: {response}")

    # Add assistant's response to history
    messages.append(ChatMessage(role="assistant", content=response.message.content))

User: How many bird species are there in Latvia?
Agent: assistant: According to the Latvian Ornithological Society (LOB), there have been **379 bird species** recorded in Latvia as of their latest updates.

This number includes regular breeders, migratory birds, and occasional visitors.
User: Which are the rarest?
Agent: assistant: Defining "rarest" can be tricky, as it can refer to species that are:
1.  **Extremely rare vagrants/accidentals:** Birds that have only been recorded once or a handful of times, far outside their normal range.
2.  **Critically endangered breeders:** Species that regularly breed in Latvia but have extremely small and declining populations.
3.  **Critically endangered migrants/winter visitors:** Species that pass through or winter in Latvia in very small numbers due to global population declines.

Based on these categories, here are some of the rarest bird species in Latvia:

### 1. Extremely Rare Vagrants/Accidentals (Single or Very Few Records)

These are of

In [7]:
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

async def stream_completion():
    # Use the async version with await
    resp = await llm.astream_complete(
        "The story of Sourcrust, the bread creature, is really interesting. It all started when..."
    )

    async for r in resp:
        print(r.delta, end="")

await stream_completion()

The story of Sourcrust, the bread creature, is really interesting. It all started when Old Man Fitzwilliam, a baker with flour permanently dusted in his eyebrows and a heart full of sourdough starter, accidentally spilled a vial of experimental yeast into his most prized batch. He'd been trying to create a yeast that could withstand the harshest winters, a yeast that could feed the whole town through even the leanest months. He hadn't expected *this*.

The dough, usually a placid, bubbling mass, began to writhe. It pulsed with an unnatural energy, expanding and contracting like a lung. Fitzwilliam, initially horrified, found himself mesmerized. He watched as the dough, now the size of a small dog, began to form rudimentary limbs. Little doughy nubs sprouted, followed by a lumpy, misshapen head.

"Good heavens," Fitzwilliam whispered, his floury hand trembling.

The creature, still doughy and pale, blinked at him with two raisin-like eyes. It let out a soft, gurgling sound, like air esc

# Prompt templates (Veidnes Uzvednēm)

In [8]:
from llama_index.core import PromptTemplate

# Define the prompt template
prompt_template = PromptTemplate("""
You are a helpful assistant.
Tell me a joke about {topic}.
""")

# Render the prompt with the input variable
topic = "cats"
rendered_prompt = prompt_template.format(topic=topic)

# Print the rendered prompt
print(rendered_prompt)


You are a helpful assistant.
Tell me a joke about cats.



In [9]:
from llama_index.core import PromptTemplate
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata
import asyncio

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)
# Define the prompt template
prompt_template = PromptTemplate("""
You are a helpful assistant.
Tell me a joke about {topic}.
""")

# Render the prompt with the input variable
topic = "cats"
rendered_prompt = prompt_template.format(topic=topic)
result = await llm.acomplete(rendered_prompt)

# Print the result
print(result)

Why did the cat join the Red Cross?

Because he wanted to be a first-aid kit!



# RAG

In [10]:
# imports
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
# get API key and create embeddings

model_name = "text-embedding-004"

embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

embeddings = embed_model.get_text_embedding("Google Gemini Embeddings.")
print(f"Dimension of embeddings: {len(embeddings)}")

Dimension of embeddings: 768


In [12]:
# Simple RAG quering txt

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

documents = SimpleDirectoryReader("/content/data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = await query_engine.aquery("What are the first programs Paul Graham tried writing?")
print(response)

The first programs Paul Graham tried writing were on the IBM 1401 that his school district used for data processing.



In [13]:
# Simple RAG quering pdf
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

documents = SimpleDirectoryReader("/content/data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = await query_engine.aquery("What are the design goals? Please give me detailes about them")
print(response)

The design goals are:

1.  **Define-and-Compose Workflows**: This allows users to create workflows by defining components and composing them into multi-agent workflows using drag-and-drop actions. It helps users understand what parameters to configure and how to configure them, providing a good developer experience. Tools are included to support authoring entities, such as defining and testing models, an IDE for generating/editing code, and a canvas-based visual layout of workflows.
2.  **Debugging and Sensemaking Tools**: These tools help users debug, interpret, and rationalize the behavior and outputs of multi-agent systems, which can be brittle and fail for multiple reasons.
3.  **Export and Deployment**: This enables the seamless export and deployment of multi-agent workflows to various platforms and environments, allowing developers to integrate the same outcomes as parts of their core applications.
4.  **Collaboration and Sharing**: This facilitates user collaboration on multi-ag

In [15]:
# Persisting the RAG
import os.path
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

Settings.llm = GoogleGenAI(
    model="gemini-2.0-flash-lite",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

query_engine = index.as_query_engine()
response = await query_engine.aquery("What are the design goals? Please give me detailes about them")
# response = await query_engine.aquery("What are the first programs Paul Graham tried writing?")
print(response)

The design goals include enabling users to define and compose workflows, providing debugging and sensemaking tools, and facilitating seamless export and deployment. Furthermore, the goals involve user collaboration on multi-agent workflow development and allowing easy sharing of creations within the community.



In [ ]:
!pip install chromadb

In [ ]:
!pip install llama-index-vector-stores-chroma

In [2]:
# Create a vectorstore
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
import os.path

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata


import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)
# load some documents
documents = SimpleDirectoryReader("/content/data").load_data()

# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

# create collection
chroma_collection = db.get_or_create_collection("quickstart")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# create your index
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

# create a query engine and query
query_engine = index.as_query_engine()
response = await query_engine.aquery("What are the first programs Paul Graham tried writing?")
print(response)

The first programs Paul Graham tried writing were on the IBM 1401 that his school district used.



In [3]:
# Query existng vectorstore

import chromadb
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata



import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)
# initialize client
db = chromadb.PersistentClient(path="./chroma_db")

# get collection
chroma_collection = db.get_or_create_collection("quickstart")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context
)

# create a query engine
query_engine = index.as_query_engine()
response = await query_engine.aquery("Which programming languages did Paul use?")
print(response)

Paul used Fortran, Lisp, Arc, and Bel.



In [4]:
import os
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata


import nest_asyncio
nest_asyncio.apply()

# Get the Google API key from Google Colab's secure storage
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Set up Gemini LLM and embedding
Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

Settings.embed_model = GoogleGenAIEmbedding(
    model_name="text-embedding-004",
    api_key=GOOGLE_API_KEY,
    title="this is a document"
)

# Use a writable path for ChromaDB (important in Google Colab)
chroma_db_path = "/content/chroma_db"
db_exists = os.path.exists(chroma_db_path)

# Initialize ChromaDB client with a persistent path
db = chromadb.PersistentClient(path=chroma_db_path)
chroma_collection = db.get_or_create_collection("quickstart")

# Assign Chroma as the vector store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build or load the index
if not db_exists:
    print("Vector store not found. Creating a new one...")
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context
    )
else:
    print("Vector store found. Loading from existing store...")
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store, storage_context=storage_context
    )

# Query the index
query_engine = index.as_query_engine()
response = await query_engine.aquery("What are the first programs Paul Graham tried writing?")
print(response)

Vector store found. Loading from existing store...
The first programs Paul Graham tried writing were on the IBM 1401 that his school district used.



# Chatbots (čatboti)

In [5]:
# Simple RAG quering pdf
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata
import os

import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)



PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# documents = SimpleDirectoryReader("/content/data").load_data()
# index = VectorStoreIndex.from_documents(documents)
chat_engine = index.as_chat_engine(chat_mode="context", llm=llm, verbose=True)
response = await chat_engine.achat("What are the design goals? Please give me detailes about them")
print(response)

The design goals of AUTO GEN STUDIO are:

1.  **Define-and-Compose Workflows:**
    *   Allow users to author workflows by defining components and composing them (via drag-and-drop actions) into multi-agent workflows.
    *   A define-and-compose workflow, where entities are first defined and persisted independently, and then composed ultimately into multi-agent workflows, provides a good developer experience.
    *   This includes providing tools to support authoring entities e.g., the ability define and test models, an IDE for generating/editing tools (code), and a canvas-based visual layout of workflows with drag-and-drop interaction for associating entities in the workflow.

2.  **Debugging and Sensemaking Tools:**
    *   Provide robust tools to help users debug, interpret, and rationalize the behavior and outputs of multi-agent systems.
    *   A critical request has been for tools to help users debug and make sense of agent responses.

3.  **Export and Deployment:**
    *   Enab

In [6]:
# Simple RAG quering pdf
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core import StorageContext, load_index_from_storage

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

# documents = SimpleDirectoryReader("/content/data").load_data()
# index = VectorStoreIndex.from_documents(documents)

PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

chat_engine = index.as_chat_engine(chat_mode="context", llm=llm, verbose=True)

while True:
    text_input = input("User: ")
    if text_input == "exit":
        break
    response = await chat_engine.achat(text_input)
    print(f"Agent: {response}")

User: What was Canadian budget?
Agent: The Canadian federal budget for the fiscal years of 2023–24 was presented to the House of Commons by Finance Minister Chrystia Freeland on 28 March 2023.

Some key figures from the budget include:

*   Total revenue: $456.8 billion (projected)
*   Total expenditures: $496.9 billion (projected)
*   Deficit: $40.1 billion (projected)
*   $43B in net new spending over six years
*   $20B for a new 15 per cent refundable tax credit to promote investment in green technologies.
*   $13B allocated to implement a means-tested dental care program.
User: Who was Paul Graham?
Agent: Based on the text provided, Paul Graham is:

*   The author of the essay from which the text is extracted.
*   The founder of Viaweb, which was later bought by Yahoo.
*   Someone who experimented with painting after selling Viaweb.
*   The person who came up with the idea for a web app for making web apps, which led to the company Aspra.
*   The creator of a new dialect of Lisp ca

In [7]:
# with memeory (ar atmiņu)
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.memory import ChatMemoryBuffer
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

# API key and LLM setup
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = GoogleGenAI(model="models/gemini-2.0-flash", api_key=GOOGLE_API_KEY)

Settings.embed_model = GoogleGenAIEmbedding(
    model_name="text-embedding-004",
    api_key=GOOGLE_API_KEY,
    title="this is a document"
)

# Storage location
PERSIST_DIR = "/content/storage"

# Load or build index
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Add memory to the chat engine
memory = ChatMemoryBuffer.from_defaults(token_limit=2000)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    llm=llm,
    memory=memory,
    verbose=True
)

# Chat loop
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    response = await chat_engine.achat(user_input)
    print(f"Agent: {response}")

User: What was Canadian budget?
Agent: The Canadian federal budget for the fiscal years of 2023–24 was presented to the House of Commons by Finance Minister Chrystia Freeland on 28 March 2023.

Some key figures from the budget include:

*   Total revenue: $456.8 billion (projected)
*   Total expenditures: $496.9 billion (projected)
*   Deficit: $40.1 billion (projected)
*   $43B in net new spending over six years
*   $20B for a new 15 per cent refundable tax credit to promote investment in green technologies.
*   $13B allocated to implement a means-tested dental care program.
User: Was total revenue bigger than expenditure?
Agent: No, the total revenue was $456.8 billion (projected), while the total expenditure was $496.9 billion (projected). Therefore, the total expenditure was bigger than the total revenue.

User: Who was Paul Graham?
Agent: Based on the provided text, Paul Graham is:

*   The author of the essay from which the text is extracted.
*   The founder of Viaweb, which was 

# Agents (Aģenti)

In [8]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

def multiply(a: int, b: int) -> int:
    """Multiply two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


agent = ReActAgent(tools=[multiply, add], llm=llm)

# Create a context to store the conversation history/session state
ctx = Context(agent)

handler = agent.run("What is 20+(2*4)?", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

```
Thought: The current language of the user is: English. I need to use the multiply tool first to calculate 2*4, then use the add tool to add the result to 20.
Action: multiply
Action Input: {"a": 2, "b": 4}
``````
Thought: Now I need to add 20 to the result of the multiplication, which is 8.
Action: add
Action Input: {'a': 20, 'b': 8}
```Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: 20+(2*4) is 28.


In [9]:
print(str(response))

20+(2*4) is 28.


In [10]:
print(response.tool_calls)

[ToolCallResult(tool_name='multiply', tool_kwargs={'a': 2, 'b': 4}, tool_id='db59ee39-2e4c-4c63-8961-f70f995382a4', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='8')], tool_name='multiply', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 4}}, raw_output=8, is_error=False), return_direct=False), ToolCallResult(tool_name='add', tool_kwargs={'a': 20, 'b': 8}, tool_id='1eebe890-4fc8-4380-8605-461a8a86838e', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='28')], tool_name='add', raw_input={'args': (), 'kwargs': {'a': 20, 'b': 8}}, raw_output=28, is_error=False), return_direct=False)]


## Agents with RAG (Aģenti ar RAG)

In [11]:
# Simple RAG quering txt

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.tools import QueryEngineTool

from llama_index.core import StorageContext, load_index_from_storage
from google.colab import userdata
import os.path

from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

# get API key and create embeddings
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)


# function tools

def multiply(a: float, b: float) -> float:
    """Multiply two numbers and returns the product"""
    return a * b

def add(a: float, b: float) -> float:
    """Add two numbers and returns the sum"""
    return a + b

# rag pipeline
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# documents = SimpleDirectoryReader("/content/data").load_data()
# index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

budget_tool = QueryEngineTool.from_defaults(
    query_engine,
    name="canadian_budget_2023",
    description="A RAG engine with some basic facts about the 2023 Canadian federal budget.",
)

agent = ReActAgent(tools=[multiply, add, budget_tool], llm=llm)

# Create a context to store the conversation history/session state
ctx = Context(agent)

handler = agent.run("What is the total amount of the 2023 Canadian federal budget multiplied by 3? Go step by step, using a tool to do any math.", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler

Thought: The current language of the user is: English. I need to first find the total amount of the 2023 Canadian federal budget and then multiply it by 3. I will use the canadian_budget_2023 tool to find the total amount and then the multiply tool to multiply it by 3.
Action: canadian_budget_2023
Action Input: {"input": "total amount of the 2023 Canadian federal budget"}
Thought: The total revenue is $456.8 billion and the total expenditure is $496.9 billion. I need to determine which one is the "total amount of the budget". Since expenditures are usually considered the total amount allocated in a budget, I will use $496.9 billion. Now I need to multiply $496.9 billion by 3.
Action: multiply
Action Input: {"a": 496.9, "b": 3}
Thought: I have the result of the multiplication. Now I can answer the question.
Answer: The total amount of the 2023 Canadian federal budget (using the total expenditure of $496.9 billion) multiplied by 3 is $1490.7 billion.


In [12]:
print(str(response))

The total amount of the 2023 Canadian federal budget (using the total expenditure of $496.9 billion) multiplied by 3 is $1490.7 billion.


## LLamaParse

In [13]:
!pip install nest_asyncio

In [14]:
from llama_parse import LlamaParse

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings

from llama_index.core import StorageContext, load_index_from_storage

from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

# Install nest_asyncio to handle nested event loops
import nest_asyncio
nest_asyncio.apply()

# LLamaParse API key

LLAMA_CLOUD_API_KEY = userdata.get('LLAMA_CLOUD_API_KEY')

# get API key and create embeddings
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

# settings
Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.1,
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)

documents = LlamaParse(result_type="markdown", api_key=LLAMA_CLOUD_API_KEY).load_data("/content/data/2023_canadian_budget.pdf")
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

response = query_engine.query("How much exactly was allocated to a tax credit to promote investment in green technologies in the 2023 Canadian federal budget?")
print(response)

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fcd15c35940>
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fcd15bd83e0>


Started parsing the file under job_id 30d9a571-f1d2-4420-8afa-9c241877fc18
In the 2023 Canadian federal budget, $20B was allocated for a new 15 per cent refundable tax credit to promote investment in green technologies.



In [ ]:
!pip install docling

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from docling.document_converter import DocumentConverter

# Initialize Docling's document converter
converter = DocumentConverter()
source = "/content/data/2023_canadian_budget.pdf"

# Convert document using Docling
result = converter.convert(source)
markdown_text = result.document.export_to_markdown()  # Get structured text

# ✅ Wrap the extracted text in a Document object
document_obj = Document(text=markdown_text)

# ✅ Pass the document object as a list to VectorStoreIndex
index = VectorStoreIndex.from_documents([document_obj])
query_engine = index.as_query_engine()

# Querying parsed document
response = query_engine2.query(
    "What was the 2023 Canadian federal budget?"
)
print(response)

## Agent Memory (Aģentu atmiņa)

In [15]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.tools import QueryEngineTool

from llama_index.core import StorageContext, load_index_from_storage
from google.colab import userdata
import os.path

from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

# get API key and create embeddings
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

Settings.llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

model_name = "text-embedding-004"

Settings.embed_model = GoogleGenAIEmbedding(
    model_name=model_name, api_key=GOOGLE_API_KEY, title="this is a document"
)


# function tools

def multiply(a: float, b: float) -> float:
    """Multiply two numbers and returns the product"""
    return a * b

def add(a: float, b: float) -> float:
    """Add two numbers and returns the sum"""
    return a + b

# rag pipeline
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("/content/data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# documents = SimpleDirectoryReader("/content/data").load_data()
# index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

budget_tool = QueryEngineTool.from_defaults(
    query_engine,
    name="canadian_budget_2023",
    description="A RAG engine with some basic facts about the 2023 Canadian federal budget.",
)

agent = ReActAgent(tools=[multiply, add, budget_tool], llm=llm)

# Create a context to store the conversation history/session state
ctx = Context(agent)

handler = agent.run("How much exactly was allocated a 'grocery rebate' in the 2023 Canadian federal budget?", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

```
Thought: The current language of the user is: English. I need to use the canadian_budget_2023 tool to find the answer to the question.
Action: canadian_budget_2023
Action Input: {"input": "grocery rebate amount"}
```Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The 2023 Canadian federal budget introduced a "grocery rebate" of up to $467 for eligible families and up to $234 for eligible single people with no kids.
The 2023 Canadian federal budget introduced a "grocery rebate" of up to $467 for eligible families and up to $234 for eligible single people with no kids.


In [16]:
handler = agent.run("How much was allocated to a implement a means-tested dental care program in the 2023 Canadian federal budget?", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

```tool_code
Thought: The current language of the user is: English. I need to use the canadian_budget_2023 tool to find the answer to the question.
Action: canadian_budget_2023
Action Input: {"input": "means-tested dental care program"}
```Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The 2023 Canadian federal budget allocated $13B to implement a means-tested dental care program.
The 2023 Canadian federal budget allocated $13B to implement a means-tested dental care program.


In [17]:
handler = agent.run("How much was the total of those two allocations added together? Use a tool to answer any questions.", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

```tool_code
Thought: The current language of the user is: English. I need to add the two amounts together. The grocery rebate is up to $467 for families and $234 for singles, so I will use the maximum amount for families. The dental care program was allocated $13B.
Action: add
Action Input: {"a": 467, "b": 13000000000}
```Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The total of those two allocations added together is $13,000,000,467.The total of those two allocations added together is $13,000,000,467.


## LlamaHub

In [18]:
!pip install llama-index-tools-yahoo-finance

In [1]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult
from llama_index.tools.yahoo_finance import YahooFinanceToolSpec
from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

def multiply(a: int, b: int) -> int:
    """Multiply two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

finance_tools = YahooFinanceToolSpec().to_tool_list()
finance_tools.extend([multiply, add])

agent = ReActAgent(tools=finance_tools, llm=llm)

# Create a context to store the conversation history/session state
ctx = Context(agent)

handler = agent.run("What is the current price of NVDA?", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

```
Thought: The current language of the user is: English. I need to use a tool to find the current price of NVDA.
Action: stock_basic_info
Action Input: {"ticker": "NVDA"}
``````tool_code
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: The current price of NVDA is 190.17 USD.
```The current price of NVDA is 190.17 USD.
```


In [2]:
! pip install llama-index-tools-tavily-research

In [1]:
from google.colab import userdata

# Set up Tavily tool
from llama_index.tools.tavily_research.base import TavilyToolSpec

# Tavily API key

TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

# Use the TAVILY_API_KEY variable instead of the string literal
tavily_tool = TavilyToolSpec(
    api_key=TAVILY_API_KEY,
)

tavily_tool_list = tavily_tool.to_tool_list()
for tool in tavily_tool_list:
    print(tool.metadata.name)
print("---------------------\n")

res = tavily_tool.search("What happened in the latest Burning Man festival?", max_results=3)
for doc in res:
    print(doc)

search
extract
---------------------

Doc ID: 54cb2760-6df4-4e29-8a0d-ddf527f5c072
Text: On a 15-acre portion of his land in Pāpaʻikou near Hilo,
Pennsylvania native Andrew Tepper held a controversial festival in
2023 and 2024 called “Falls on Fire,” an event with a large wooden
effigy inspired by the annual weeklong, large-scale Burning Man held
in the Black Rock Desert in Nevada. [...] Despite repeated warnings to
not to hold the ...
Doc ID: 1d60b13c-0272-4bcb-b1e6-989fd11d0adb
Text: 2025 Burning Man in Black Rock City, Nevada. This afterburn ...
highlights the stunning art installations, sculptures, mutant
Doc ID: b7b7de40-4563-4740-9676-6784c73ab897
Text: The 2026 Burning Man theme has arrived! Expressed by the cosmic
tree that links us to one another, to the unseen, and unknowable, Axis
Mundi celebrates our interconnectedness, the new social realities we
are shaping together, and our enduring relationship with the natural
world. What does Axis Mundi inspire YOU to create?  ## Impo

In [2]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult
from llama_index.tools.tavily_research.base import TavilyToolSpec

from llama_index.llms.google_genai import GoogleGenAI
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# get API key and create embeddings

llm = GoogleGenAI(
    model="models/gemini-2.0-flash",
    api_key=GOOGLE_API_KEY
)

# Tavily API key
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

# Use the TAVILY_API_KEY variable instead of the string literal
tavily_tool = TavilyToolSpec(
    api_key=TAVILY_API_KEY,
)

tavily_tool_list = tavily_tool.to_tool_list()
for tool in tavily_tool_list:
    print(tool.metadata.name)

agent = ReActAgent(tools=tavily_tool_list, llm=llm)

# Create a context to store the conversation history/session state
ctx = Context(agent)

handler = agent.run("Write a deep analysis in markdown syntax about the latest burning man floods", ctx=ctx)

async for ev in handler.stream_events():
    # if isinstance(ev, ToolCallResult):
    #     print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await handler
print(str(response))

search
extract
```
Thought: The current language of the user is: English. I need to use a tool to find information about the latest Burning Man floods.
Action: search
Action Input: {"query": "Burning Man 2023 floods", "max_results": 5}
```
Thought: The current language of the user is: English. I have some information about the Burning Man floods of 2023. I will use this information to write a deep analysis in markdown syntax.
Answer: # Deep Analysis of the Burning Man 2023 Floods

Burning Man 2023, the 35th annual gathering in Nevada's Black Rock Desert (August 27 - September 4, 2023), experienced significant disruption due to heavy rainfall and subsequent flooding. An estimated 73,000 attendees were impacted by the event, which led to a temporary shutdown of the festival, shelter-in-place advisories, and challenging conditions for participants.

## The Deluge

On September 1, 2023, the Black Rock Desert was hit by substantial rainfall. The National Weather Service issued a flash flood

# Workflows (Darba plūsmas)

In [ ]:
! pip install llama-index-utils-workflow

In [1]:
# Single-step workflow

from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # do something here
        return StopEvent(result="Hello, world!")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run()
print(result)

Hello, world!


In [26]:
from llama_index.utils.workflow import draw_all_possible_flows

w = MyWorkflow()

draw_all_possible_flows(w, filename="basic_workflow.html")

basic_workflow.html


In [5]:
# Multi-step workfow

from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Event,
)

# Define custom events classes that inherit from Event class

class FirstEvent(Event):
    first_output: str


class SecondEvent(Event):
    second_output: str

# Define the workflow that inherits from Workflow class

class MyWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> FirstEvent:
        print(ev.first_input)
        return FirstEvent(first_output="First step complete.")

    @step
    async def step_two(self, ev: FirstEvent) -> SecondEvent:
        print(ev.first_output)
        return SecondEvent(second_output="Second step complete.")

    @step
    async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run(first_input="Start the workflow.")
print(result)

Start the workflow.
First step complete.
Second step complete.
Workflow complete.


In [25]:
# visualise the workflow

from llama_index.utils.workflow import draw_all_possible_flows

w = MyWorkflow()
draw_all_possible_flows(w, filename="multi_step_workflow.html")

multi_step_workflow.html


## Branching and Looping (Zarošanās un cikli)

### Loop flow (Cikliska  darba plūsma)

In [9]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Event,
)
import random

# Define events

class LoopEvent(Event):
    loop_output: str

class FirstEvent(Event):
    first_output: str

# Defie workfow

class LoopWorkflow(Workflow):
  @step
  async def step_one(self, ev: StartEvent | LoopEvent) -> FirstEvent | LoopEvent:
      if random.randint(0, 1) == 0:
          print("Bad thing happened")
          return LoopEvent(loop_output="Back to step one.")
      else:
          print("Good thing happened")
          return FirstEvent(first_output="First step complete.")
  @step
  async def step_two(self, ev: FirstEvent) -> SecondEvent:
        print(ev.first_output)
        return SecondEvent(second_output="Second step complete.")

  @step
  async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")


w = LoopWorkflow(timeout=10, verbose=False)
result = await w.run(first_input="Start the workflow.")
print(result)

Good thing happened
First step complete.
Second step complete.
Workflow complete.


In [24]:
from llama_index.utils.workflow import draw_all_possible_flows

# Instantiate the workflow
w = LoopWorkflow()

# Then draw
draw_all_possible_flows(w, filename="loop_workflow.html")


loop_workflow.html


In [15]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Event,
)
import random

class FailedEvent(Event):
    error: str

class QueryEvent(Event):
    query: str

class LoopExampleFlow(Workflow):

    @step()
    async def answer_query(self, ev: StartEvent | QueryEvent ) -> FailedEvent | StopEvent:
        query = ev.query
        # try to answer the query
        random_number = random.randint(0, 1)
        if (random_number == 0):
            return FailedEvent(error="Failed to answer the query.")
        else:
            return StopEvent(result="The answer to your query")

    @step()
    async def improve_query(self, ev: FailedEvent) -> QueryEvent | StopEvent:
        # improve the query or decide it can't be fixed
        random_number = random.randint(0, 1)
        if (random_number == 0):
            return QueryEvent(query="Here's a better query.")
        else:
            return StopEvent(result="Your query can't be fixed.")

l = LoopExampleFlow(timeout=10, verbose=True)
result = await l.run(query="What's LlamaIndex?")
print(result)

The answer to your query


In [27]:
from llama_index.utils.workflow import draw_all_possible_flows

w = LoopExampleFlow()

draw_all_possible_flows(w, filename="loop_example_workflow.html")

loop_example_workflow.html


### Branching flow (Zarošanās plūsmaplūsma)

In [23]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Event,
)

class BranchA1Event(Event):
    payload: str


class BranchA2Event(Event):
    payload: str


class BranchB1Event(Event):
    payload: str


class BranchB2Event(Event):
    payload: str


class BranchWorkflow(Workflow):
    @step
    async def start(self, ev: StartEvent) -> BranchA1Event | BranchB1Event:
        if random.randint(0, 1) == 0:
            print("Go to branch A")
            return BranchA1Event(payload="Branch A")
        else:
            print("Go to branch B")
            return BranchB1Event(payload="Branch B")

    @step
    async def step_a1(self, ev: BranchA1Event) -> BranchA2Event:
        print(ev.payload)
        return BranchA2Event(payload=ev.payload)

    @step
    async def step_b1(self, ev: BranchB1Event) -> BranchB2Event:
        print(ev.payload)
        return BranchB2Event(payload=ev.payload)

    @step
    async def step_a2(self, ev: BranchA2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch A complete.")

    @step
    async def step_b2(self, ev: BranchB2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch B complete.")

b = BranchWorkflow(timeout=10, verbose=True)
result = await b.run(query="What's LlamaIndex?")
print(result)

Go to branch A
Branch A
Branch A
Branch A complete.


In [28]:
from llama_index.utils.workflow import draw_all_possible_flows

w = BranchWorkflow()

draw_all_possible_flows(w, filename="branch_workflow.html")

branch_workflow.html
